# 02 — Feature Engineering
**TF-IDF Vectorization + BERT Embeddings + Word2Vec**

This notebook:

1. Builds TF-IDF feature matrices (fit on train, transform test)
2. Generates BERT sentence embeddings via `sentence-transformers`
3. Trains Word2Vec and generates mean-pooled document embeddings
4. Saves all matrices + the fitted vectorizer / model to `../data/features/`

In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 86.1 MB/s eta 0:00:00


## Imports & Config

In [2]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib
from tqdm.auto import tqdm
# Word2Vec
from gensim.models import Word2Vec

In [3]:
import os
from google.colab import drive

drive.mount('/content/drive')
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/SmartReviewAnalyzer'


#Paths
PROCESSED_DIR = os.path.join(DRIVE_PROJECT_ROOT, 'data/processed')
FEATURES_DIR  = os.path.join(DRIVE_PROJECT_ROOT, 'data/features')

TRAIN_CSV = os.path.join(PROCESSED_DIR, 'preprocessed_train.csv')
TEST_CSV  = os.path.join(PROCESSED_DIR, 'preprocessed_test.csv')

os.makedirs(FEATURES_DIR, exist_ok=True)



Mounted at /content/drive


## Load Preprocessed Data

In [4]:
df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

print(f'Train : {df_train.shape}')
print(f'Test  : {df_test.shape}')
df_train.head(2)

Train : (519575, 4)
Test  : (39972, 4)


,binary_label,text,cleaned_text_tfidf,cleaned_text_bert
0,1,dr. goldberg offers everything i look for in a...,dr goldberg offer everything look general prac...,dr. goldberg offers everything i look for in a...
1,0,"Unfortunately, the frustration of being Dr. Go...",unfortunately frustration dr goldberg patient ...,"Unfortunately, the frustration of being Dr. Go..."


In [5]:
# Separate text columns and labels
train_text_tfidf = df_train['cleaned_text_tfidf'].fillna('').tolist()
test_text_tfidf  = df_test['cleaned_text_tfidf'].fillna('').tolist()

train_text_bert  = df_train['cleaned_text_bert'].fillna('').tolist()
test_text_bert   = df_test['cleaned_text_bert'].fillna('').tolist()

y_train = df_train['binary_label'].values
y_test  = df_test['binary_label'].values

print(f'Train texts (TF-IDF) : {len(train_text_tfidf):,}')
print(f'Test  texts (TF-IDF) : {len(test_text_tfidf):,}')
print(f'Train texts (BERT)   : {len(train_text_bert):,}')
print(f'Test  texts (BERT)   : {len(test_text_bert):,}')

Train texts (TF-IDF) : 519,575
Test  texts (TF-IDF) : 39,972
Train texts (BERT)   : 519,575
Test  texts (BERT)   : 39,972


---
## Word2Vec Embeddings

Using **`gensim`'s Word2Vec** trained directly on the review corpus:
- Learns dense word vectors from scratch on domain-specific text
- Document embeddings = mean-pooled word vectors (fast & effective)
- **300-dimensional** embeddings, complementary to TF-IDF and BERT
- Captures semantic similarity not present in bag-of-words features

### Config

In [6]:
# Word2Vec hyper-parameters
W2V_VECTOR_SIZE = 300       # embedding dimensionality
W2V_WINDOW      = 5         # context window size
W2V_MIN_COUNT   = 2         # ignore words with freq < this
W2V_WORKERS     = 4         # parallel training threads
W2V_EPOCHS      = 10        # training epochs
W2V_SG          = 1         # 1 = skip-gram, 0 = CBOW

print('Word2Vec config ready.')
print(f'  vector_size : {W2V_VECTOR_SIZE}')
print(f'  window      : {W2V_WINDOW}')
print(f'  min_count   : {W2V_MIN_COUNT}')
print(f'  sg (skip-g) : {W2V_SG}')
print(f'  epochs      : {W2V_EPOCHS}')


Word2Vec config ready.
  vector_size : 300
  window      : 5
  min_count   : 2
  sg (skip-g) : 1
  epochs      : 10


### Tokenise Text for Word2Vec

Word2Vec expects a list of token lists. We reuse `cleaned_text_tfidf` (already lowercased / stripped) and do a simple whitespace split — no lemmatisation needed here.

In [7]:
# Tokenise — simple whitespace split on the already-cleaned text
train_tokens_w2v = [text.split() for text in train_text_tfidf]
test_tokens_w2v  = [text.split() for text in test_text_tfidf]

# Quick sanity
print(f'Train documents : {len(train_tokens_w2v):,}')
print(f'Test  documents : {len(test_tokens_w2v):,}')
print(f'Example tokens  : {train_tokens_w2v[0][:10]}')


Train documents : 519,575
Test  documents : 39,972
Example tokens  : ['dr', 'goldberg', 'offer', 'everything', 'look', 'general', 'practitioner', 'nice', 'easy', 'talk']


### Train Word2Vec Model

In [8]:
from gensim.models import Word2Vec

print('Training Word2Vec on training corpus ...')
w2v_model = Word2Vec(
    sentences   = train_tokens_w2v,
    vector_size = W2V_VECTOR_SIZE,
    window      = W2V_WINDOW,
    min_count   = W2V_MIN_COUNT,
    workers     = W2V_WORKERS,
    epochs      = W2V_EPOCHS,
    sg          = W2V_SG,
    seed        = 42,
)

vocab_size = len(w2v_model.wv)
print(f'Training complete.')
print(f'  Vocabulary size : {vocab_size:,} words')
print(f'  Embedding dim   : {w2v_model.wv.vector_size}')

# Quick sanity — most similar words to 'good'
if 'good' in w2v_model.wv:
    print('\nMost similar to "good":')
    for word, sim in w2v_model.wv.most_similar('good', topn=5):
        print(f'  {word:<20s} {sim:.4f}')


Training Word2Vec on training corpus ...
Training complete.
  Vocabulary size : 89,744 words
  Embedding dim   : 300

Most similar to "good":
  great                0.7149
  decent               0.7118
  tasty                0.6867
  blehhhhh             0.6611
  gooder               0.6546


### Build Document Embeddings (Mean Pooling)

Each document embedding is the **mean of its in-vocabulary word vectors**. Out-of-vocabulary (OOV) tokens are skipped; documents with no known tokens fall back to a zero vector.

In [9]:
import numpy as np

def mean_pool_w2v(token_lists, wv, vector_size):
    """Return (n_docs, vector_size) float32 array via mean pooling."""
    embeddings = np.zeros((len(token_lists), vector_size), dtype=np.float32)
    for i, tokens in enumerate(token_lists):
        vecs = [wv[t] for t in tokens if t in wv]
        if vecs:
            embeddings[i] = np.mean(vecs, axis=0)
    return embeddings

print(f'Building train embeddings ({len(train_tokens_w2v):,} docs) ...')
X_train_w2v = mean_pool_w2v(train_tokens_w2v, w2v_model.wv, W2V_VECTOR_SIZE)
print(f'Train W2V matrix : {X_train_w2v.shape}  dtype={X_train_w2v.dtype}')

print(f'Building test  embeddings ({len(test_tokens_w2v):,} docs) ...')
X_test_w2v = mean_pool_w2v(test_tokens_w2v, w2v_model.wv, W2V_VECTOR_SIZE)
print(f'Test  W2V matrix : {X_test_w2v.shape}  dtype={X_test_w2v.dtype}')

# Sanity — L2 norms (not normalised, just informational)
norms = np.linalg.norm(X_train_w2v[:100], axis=1)
print(f'\nEmbedding L2 norms (first 100 train rows):')
print(f'  mean={norms.mean():.4f}  min={norms.min():.4f}  max={norms.max():.4f}')

# Count zero vectors (OOV docs)
zero_rows = (X_train_w2v.sum(axis=1) == 0).sum()
print(f'  Zero-vector (OOV) docs in train : {zero_rows} ({100*zero_rows/len(X_train_w2v):.2f}%)')


Building train embeddings (519,575 docs) ...
Train W2V matrix : (519575, 300)  dtype=float32
Building test  embeddings (39,972 docs) ...
Test  W2V matrix : (39972, 300)  dtype=float32

Embedding L2 norms (first 100 train rows):
  mean=1.5761  min=1.4007  max=2.2432
  Zero-vector (OOV) docs in train : 2 (0.00%)


### Save Word2Vec Features

In [10]:
w2v_train_path = os.path.join(FEATURES_DIR, 'X_train_w2v.npy')
w2v_test_path  = os.path.join(FEATURES_DIR, 'X_test_w2v.npy')
w2v_model_path = os.path.join(FEATURES_DIR, 'word2vec.model')

np.save(w2v_train_path, X_train_w2v)
np.save(w2v_test_path,  X_test_w2v)
w2v_model.save(w2v_model_path)          # saves full model (can resume training)

print('Word2Vec artefacts saved:')
for p in [w2v_train_path, w2v_test_path, w2v_model_path]:
    size_mb = os.path.getsize(p) / 1_048_576
    print(f'  {p}  ({size_mb:.2f} MB)')

# To reload later:
#   from gensim.models import Word2Vec
#   w2v_model = Word2Vec.load(w2v_model_path)
#   X_train_w2v = np.load(w2v_train_path)


Word2Vec artefacts saved:
  /content/drive/MyDrive/SmartReviewAnalyzer/data/features/X_train_w2v.npy  (594.61 MB)
  /content/drive/MyDrive/SmartReviewAnalyzer/data/features/X_test_w2v.npy  (45.74 MB)
  /content/drive/MyDrive/SmartReviewAnalyzer/data/features/word2vec.model  (2.98 MB)
